# Bruise Segmentation — Inference, Benchmarking & Evaluation Demo

This notebook runs our **5 trained models** and answers two questions:

1. **How fast are they?** (three different benchmarks — they measure different things)
2. **How accurate are they?** (Dice / IoU / miss-rate on the 185-image test set)

**Everything is written out in this notebook.** We deliberately do *not* call
`pipeline/benchmark_640.py` for the timing code — the timing loops are spelled out
in plain cells below so you can read exactly what is being measured. We only import
from `pipeline/` for the boring, well-tested parts (dataset loading, checkpoint
loading, metrics).

---

## The three benchmarks at a glance

"Speed" has no single answer — it depends entirely on where you start and stop the
stopwatch. So we report three:

| # | Name | Stopwatch starts | Stopwatch stops | Answers |
|---|------|------------------|-----------------|---------|
| **1** | Raw forward (resize **NOT** timed) | tensor already on GPU | 640 mask on CPU | *How fast is the architecture?* |
| **2** | Full 640 pipeline (resize **timed**) | RGB image in RAM | 640 mask on CPU | *How fast is inference at 640?* |
| **3** | **Deployment** (recommended) | RGB image in RAM | mask at **native 4022×6024** | *How fast is the real product?* |

**Why benchmark 3 exists.** Benchmarks 1 and 2 both stop at a 640×640 mask. But the
thing we actually ship is a mask overlaid on the **original camera photo**, which is
4022×6024 = **24.2M pixels — 59× more than 640×640**. Upsampling the mask back up is
real work that neither 1 nor 2 counts. Benchmark 3 measures the whole camera-frame-to-
overlayable-mask path, which is the only number a deployment decision should use.

## 1. Mount Google Drive

The trained models and test images live in a zip on Drive
(`bruise_colab_gpu_full.zip`, ~868 MB). We mount Drive to get at it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bruise_segmentation_gpu'
ZIP_PATH  = f'{DRIVE_DIR}/bruise_colab_gpu_full.zip'

import os
assert os.path.exists(ZIP_PATH), f'{ZIP_PATH} not found — upload the zip to {DRIVE_DIR}/ first.'
print('Found package:', ZIP_PATH, f'({os.path.getsize(ZIP_PATH)/1e6:.0f} MB)')

Mounted at /content/drive
Found package: /content/drive/MyDrive/bruise_segmentation_gpu/bruise_colab_gpu_full.zip (910 MB)


## 2. Check we actually have a GPU

Timing numbers are meaningless without one. This cell stops the notebook immediately
if the runtime is CPU-only, rather than letting you collect nonsense.

**Runtime → Change runtime type → A100 GPU** if this fails.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > A100 GPU, then re-run.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

Wed Jul 15 14:46:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 3. Copy the zip to local disk and unzip

We copy off Drive first. Reading images directly from the Drive mount goes over the
network, and that unpredictable delay would leak into the timings we're about to
measure. Local disk keeps the benchmark honest.

In [ ]:
import shutil, time
LOCAL_DIR = '/content/bruise_gpu'

t0 = time.time()
shutil.copy(ZIP_PATH, '/content/pkg.zip')
print(f'Copied in {time.time()-t0:.0f}s')

!rm -rf {LOCAL_DIR}
!mkdir -p {LOCAL_DIR}
t0 = time.time()
!unzip -q /content/pkg.zip -d {LOCAL_DIR}
print(f'Unzipped in {time.time()-t0:.0f}s')

Copied in 33s
Unzipped in 12s


## 4. Install the libraries

Colab already ships torch + CUDA, so we never reinstall torch (that would break the
GPU build). We only add the four libraries our pipeline needs.

In [ ]:
%cd {LOCAL_DIR}
!pip install -q transformers ultralytics albumentations opencv-python-headless

/content/bruise_gpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.8 MB/s eta 0:00:00


## 5. Imports and configuration

Note the import order: **`cv2` before `ultralytics`**. On some setups importing
ultralytics first makes `cv2.imread(..., IMREAD_GRAYSCALE)` return shape `(H,W,1)`
instead of `(H,W)`, which silently corrupts mask handling. Cheap to avoid.

In [ ]:
import cv2                       # before ultralytics, on purpose
import numpy as np
import pandas as pd
import torch, torch.nn.functional as F
import statistics, sys, json
from torch.utils.data import DataLoader
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox

sys.path.insert(0, LOCAL_DIR)
from pipeline.io_utils import load_yaml
from pipeline.data import BruiseDataset, load_fixed_test
from pipeline.models import load_segformer_model, count_params
from pipeline.metrics import compute_image_row

paths = load_yaml('configs/paths.yaml')
cfg   = load_yaml('configs/common_train.yaml')
IMG_H = IMG_W = 640                      # every model runs and is scored at 640x640
assert (cfg['img_h'], cfg['img_w']) == (IMG_H, IMG_W)

torch.backends.cudnn.benchmark = True    # autotune conv kernels; safe, input shape is fixed
print('Config loaded. Model grid:', IMG_H, 'x', IMG_W)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Config loaded. Model grid: 640 x 640


## 6. Load the test dataset

`BruiseDataset` (from `pipeline/data.py`) is the same loader used during training. For
each of the 185 test images it:

1. reads the photo (native **4022×6024**) and its ground-truth mask,
2. **resizes both to 640×640** — image bilinear, mask nearest-neighbour (nearest keeps
   the mask strictly 0/1; bilinear would invent grey half-pixels),
3. applies **ImageNet normalisation** to the image,
4. returns tensors.

Because prediction and ground truth land on the *same* 640 grid, Dice is a fair
comparison. `batch_size=1` because we want per-image numbers, not batch averages.

In [ ]:
test_df = load_fixed_test(paths['fixed_test_manifest'])
test_ds = BruiseDataset(test_df, IMG_H, IMG_W, training=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

x, y, stem, img_path, mask_path = test_ds[0]
print(f'Test images      : {len(test_df)}')
print(f'Image tensor     : {tuple(x.shape)}  (3 channels, 640x640, ImageNet-normalised)')
print(f'GT mask tensor   : {tuple(y.shape)}  (1 channel, 640x640, values 0/1)')
print(f'Native photo size: {cv2.imread(str(img_path)).shape}  <-- what the camera gives us')

Test images      : 185
Image tensor     : (3, 640, 640)  (3 channels, 640x640, ImageNet-normalised)
GT mask tensor   : (1, 640, 640)  (1 channel, 640x640, values 0/1)
Native photo size: (4022, 6024, 3)  <-- what the camera gives us


## 7. Load the 5 trained models

Two different loaders, because the two families were trained in different frameworks:

* **SegFormer** (B2 teacher, B0 direct, B0 distilled) — `load_segformer_model()` reads
  `best_model.pt` and **also returns that model's threshold**. The threshold was chosen
  on the *validation* set during training and is frozen; we never tune it on test.
  Prediction = `sigmoid(logit) >= threshold`.
* **YOLO** (direct, distilled) — loaded through Ultralytics. YOLO has **no threshold**:
  its native postprocessing takes an `argmax` over the 2 class channels, which fixes the
  operating point by construction.

In [ ]:
import contextlib, warnings
from transformers.utils import logging as hf_logging

@contextlib.contextmanager
def quiet_pretrained_load():
    """HF reports the decode_head as newly initialised because we start from an
    ImageNet classification checkpoint; load_segformer_model() then loads our
    fine-tuned weights over it with strict key matching. Errors still raise."""
    prev = hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()
    hf_logging.disable_progress_bar()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            yield
    finally:
        hf_logging.set_verbosity(prev)
        hf_logging.enable_progress_bar()
MODELS = {}

# --- SegFormer: returns a validation-selected threshold with each model ---
for run, disp, pkey in [
    ('segformer_b2_teacher',   'SegFormer-B2 (teacher)',   'segformer_b2_pretrained'),
    ('segformer_b0_direct',    'SegFormer-B0 (direct)',    'segformer_b0_pretrained'),
    ('segformer_b0_distilled', 'SegFormer-B0 (distilled)', 'segformer_b0_pretrained'),
]:
    with quiet_pretrained_load():
        model, threshold, ckpt = load_segformer_model(run, pkey, paths, DEVICE)
    model = model.to(DEVICE).eval()
    MODELS[run] = {'display': disp, 'family': 'segformer', 'model': model,
                   'threshold': threshold, 'params_m': count_params(model)[0]/1e6}

# --- YOLO: wrapper (for native .predict) + raw module (for timing the forward pass) ---
import copy
for run, disp in [('yolo_sem_direct', 'YOLO26n-sem (direct)'),
                  ('yolo_sem_distilled', 'YOLO26n-sem (distilled)')]:
    wrapper = YOLO(f'{run}/ultralytics_runs/train/weights/best.pt')
    nn_model = copy.deepcopy(wrapper.model).to(DEVICE).eval()
    MODELS[run] = {'display': disp, 'family': 'yolo', 'wrapper': wrapper, 'model': nn_model,
                   'threshold': None,
                   'params_m': sum(p.numel() for p in nn_model.parameters())/1e6}

for k, v in MODELS.items():
    thr = 'argmax (no threshold)' if v['threshold'] is None else f"threshold = {v['threshold']:.2f}"
    print(f"{v['display']:26s} {v['params_m']:6.2f}M params | {thr}")

SegFormer-B2 (teacher)      27.35M params | threshold = 0.75
SegFormer-B0 (direct)        3.71M params | threshold = 0.60
SegFormer-B0 (distilled)     3.71M params | threshold = 0.55
YOLO26n-sem (direct)         1.63M params | argmax (no threshold)
YOLO26n-sem (distilled)      1.63M params | argmax (no threshold)


## 8. Preprocessing and postprocessing — spelled out here

**This is the most important cell to read.** The two model families need *different*
preprocessing, because they were trained differently:

| | SegFormer | YOLO |
|---|---|---|
| Resize | stretch to 640×640 | **letterbox** (keep aspect ratio, pad) |
| Scaling | **ImageNet** mean/std | plain **/255** |
| Mask from output | `sigmoid(logit) >= threshold` | `argmax` over 2 channels |

**Why this matters so much:** we once fed YOLO an ImageNet-normalised tensor (SegFormer's
recipe). It didn't crash — it just quietly scored **0.506 mean Dice instead of 0.735**.
A model only works on the input distribution it was trained on. Getting this wrong is
silent, so we write both paths out explicitly rather than sharing one function.

In [ ]:
"""comment the letter box - modular inference code."""

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
_letterbox = LetterBox(new_shape=(IMG_H, IMG_W))

def preprocess_segformer(img_rgb):
    """Native RGB photo -> [1,3,640,640] GPU tensor, ImageNet-normalised (stretch resize)."""
    r = cv2.resize(img_rgb, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)
    r = (r.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(r).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

def preprocess_yolo(img_rgb):
    """Native RGB photo -> [1,3,640,640] GPU tensor, /255 only (letterbox resize)."""
    r = _letterbox(image=img_rgb)                       # aspect-preserving + padding
    t = torch.from_numpy(r).permute(2, 0, 1).float().div(255.0).unsqueeze(0)
    return t.to(DEVICE)

def postprocess_segformer(logits, threshold):
    """[1,1,640,640] logits -> 640x640 uint8 mask on CPU."""
    prob = torch.sigmoid(logits[:, 0])
    return (prob >= threshold).to(torch.uint8)[0].cpu().numpy()

def postprocess_yolo(preds):
    """[1,2,640,640] class logits -> 640x640 uint8 mask on CPU (argmax = native rule)."""
    if isinstance(preds, (tuple, list)):
        preds = preds[0]
    if preds.shape[-2:] != (IMG_H, IMG_W):
        preds = F.interpolate(preds.float(), size=(IMG_H, IMG_W), mode='bilinear', align_corners=False)
    return preds.argmax(dim=1).to(torch.uint8)[0].cpu().numpy()
    """use dataloader for testing as well"""

print('Preprocessing / postprocessing defined for both families.')

Preprocessing / postprocessing defined for both families.


## 9. The timing harness

Two details that make GPU timing correct, and are easy to get wrong:

* **`torch.cuda.synchronize()`** — the GPU runs *asynchronously*. Without this we'd be
  timing how long it takes to *queue* the work (microseconds), not to *do* it
  (milliseconds). We sync before starting and before stopping the clock.
* **Warmup** — the first few calls pay one-off costs (kernel selection, memory
  allocation). We run 30 throwaway iterations first so we measure steady state.

We report **median** and **p95** alongside the mean. The mean gets dragged around by
occasional slow frames; the median is the typical case and p95 is the "how bad does it
get" number that actually matters for a responsive UI.

In [ ]:
N_WARMUP, N_REPEATS = 30, 100

def time_it(fn, label=''):
    """Run fn() N_REPEATS times, return timing stats in ms. GPU-correct."""
    for _ in range(N_WARMUP):          # warmup: don't measure one-off startup costs
        fn()
    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)

    times = []
    for _ in range(N_REPEATS):
        torch.cuda.synchronize(DEVICE)          # make sure GPU is idle before we start
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize(DEVICE)          # wait for GPU to actually finish
        times.append((time.perf_counter() - t0) * 1000.0)

    times = np.array(times)
    return {'mean_ms': times.mean(), 'median_ms': np.median(times),
            'p95_ms': np.percentile(times, 95), 'fps': 1000.0 / times.mean(),
            'peak_gpu_mb': torch.cuda.max_memory_allocated(DEVICE) / 1024**2}

# One real photo, decoded once, reused by every benchmark below.
BENCH_IMG = cv2.cvtColor(cv2.imread(str(test_df.iloc[0]['image_path'])), cv2.COLOR_BGR2RGB)
NATIVE_H, NATIVE_W = BENCH_IMG.shape[:2]
print(f'Benchmark photo: {NATIVE_W}x{NATIVE_H} ({NATIVE_H*NATIVE_W/1e6:.1f}M pixels)')
print(f'Protocol: {N_WARMUP} warmup + {N_REPEATS} timed reps, cuda.synchronize() around each')

Benchmark photo: 6024x4022 (24.2M pixels)
Protocol: 30 warmup + 100 timed reps, cuda.synchronize() around each


---
# Benchmark 1 — Raw forward pass (resize **NOT** timed)

The input tensor is built **once, before the stopwatch starts**, and is already sitting
on the GPU. We time only the network and turning its output into a mask.

###  INCLUDED in the timer
- model forward pass
- sigmoid + threshold (SegFormer) / argmax (YOLO)
- copying the 640 mask back to CPU

###  EXCLUDED from the timer
- reading/decoding the photo from disk
- **resize to 640** ← the headline exclusion
- normalisation / scaling
- moving the tensor to the GPU
- resizing the mask back to native resolution

**Use this for:** comparing architectures fairly. **Don't use this for:** promising
anyone a frame rate — it ignores most of the real work.

In [ ]:
bench1 = {}
for run, m in MODELS.items():
    if m['family'] == 'segformer':
        x_gpu = preprocess_segformer(BENCH_IMG)          # built BEFORE the timer
        def fn(mm=m['model'], t=m['threshold'], xx=x_gpu):
            with torch.no_grad():
                postprocess_segformer(mm(xx), t)
    else:
        x_gpu = preprocess_yolo(BENCH_IMG)               # built BEFORE the timer
        def fn(mm=m['model'], xx=x_gpu):
            with torch.no_grad():
                postprocess_yolo(mm(xx))
    bench1[run] = time_it(fn)
    b = bench1[run]
    print(f"{m['display']:26s} {b['median_ms']:7.2f} ms (median) | {b['fps']:6.1f} FPS | peak {b['peak_gpu_mb']:6.1f} MB")

SegFormer-B2 (teacher)       33.67 ms (median) |   29.7 FPS | peak  875.8 MB
SegFormer-B0 (direct)        16.93 ms (median) |   59.0 FPS | peak  419.7 MB
SegFormer-B0 (distilled)     16.80 ms (median) |   59.3 FPS | peak  419.8 MB
YOLO26n-sem (direct)          8.48 ms (median) |  117.2 FPS | peak  208.7 MB
YOLO26n-sem (distilled)       8.51 ms (median) |  117.0 FPS | peak  208.7 MB


---
# Benchmark 2 — Full 640 pipeline (resize **IS** timed)

Now the stopwatch starts from a **decoded photo sitting in RAM** — which is what a camera
hands you. Everything needed to turn that into a 640 mask is timed.

###  INCLUDED in the timer
- **resize to 640** ← now counted
- normalisation (ImageNet) / scaling (/255)
- moving the tensor to the GPU
- model forward pass
- sigmoid + threshold / argmax
- copying the 640 mask back to CPU

###  EXCLUDED from the timer
- reading/decoding the photo from disk (a camera gives you the frame already in memory;
  disk speed measures your SSD, not your model)
- resizing the mask back to native resolution

**Use this for:** honest inference cost at 640. **Still missing:** the final upsample —
which is what benchmark 3 adds.

In [ ]:
bench2 = {}
for run, m in MODELS.items():
    if m['family'] == 'segformer':
        def fn(mm=m['model'], t=m['threshold']):
            with torch.no_grad():
                xx = preprocess_segformer(BENCH_IMG)     # resize+norm+H2D now INSIDE the timer
                postprocess_segformer(mm(xx), t)
    else:
        def fn(mm=m['model']):
            with torch.no_grad():
                xx = preprocess_yolo(BENCH_IMG)          # resize+/255+H2D now INSIDE the timer
                postprocess_yolo(mm(xx))
    bench2[run] = time_it(fn)
    b = bench2[run]
    print(f"{m['display']:26s} {b['median_ms']:7.2f} ms (median) | {b['fps']:6.1f} FPS | peak {b['peak_gpu_mb']:6.1f} MB")

SegFormer-B2 (teacher)       44.96 ms (median) |   22.1 FPS | peak  880.5 MB
SegFormer-B0 (direct)        26.96 ms (median) |   36.6 FPS | peak  422.5 MB
SegFormer-B0 (distilled)     27.12 ms (median) |   36.7 FPS | peak  422.5 MB
YOLO26n-sem (direct)         12.47 ms (median) |   80.2 FPS | peak  213.4 MB
YOLO26n-sem (distilled)      12.23 ms (median) |   80.6 FPS | peak  213.4 MB


---
# Benchmark 3 — Deployment: camera frame → overlayable mask ⭐ **recommended**

### Why this one is the right number

Benchmarks 1 and 2 both stop at a **640×640 mask**. But nobody wants a 640 mask — the
product is a mask drawn on the **original photo**, which is **4022×6024 = 24.2M pixels,
59× larger** than the model's grid. Upsampling the mask back up is genuine work that
neither benchmark above counts.

So benchmark 3 times the complete path a real system runs per frame, and reports
**median and p95** rather than only the mean — p95 is what a user actually perceives as
"sometimes it lags."

###INCLUDED in the timer
- resize to 640, normalisation/scaling, transfer to GPU
- model forward pass
- sigmoid + threshold / argmax
- mask back to CPU
- **upsample the mask 640 → 4022×6024** (nearest-neighbour, to keep it binary) ← the addition

###EXCLUDED from the timer
- reading/decoding the photo from disk (camera frames arrive in RAM)
- loading the model (once at startup, not per frame)
- drawing the coloured overlay (a UI concern, not a model concern)

**Use this for:** any claim about deployment frame rate.

In [ ]:
def upsample_to_native(mask640):
    """640 mask -> native camera resolution. INTER_NEAREST keeps it strictly 0/1."""
    return cv2.resize(mask640, (NATIVE_W, NATIVE_H), interpolation=cv2.INTER_NEAREST)

bench3 = {}
for run, m in MODELS.items():
    if m['family'] == 'segformer':
        def fn(mm=m['model'], t=m['threshold']):
            with torch.no_grad():
                xx  = preprocess_segformer(BENCH_IMG)
                msk = postprocess_segformer(mm(xx), t)
                upsample_to_native(msk)                  # <-- the step 1 and 2 both skip
    else:
        def fn(mm=m['model']):
            with torch.no_grad():
                xx  = preprocess_yolo(BENCH_IMG)
                msk = postprocess_yolo(mm(xx))
                upsample_to_native(msk)                  # <-- the step 1 and 2 both skip
    bench3[run] = time_it(fn)
    b = bench3[run]
    print(f"{m['display']:26s} {b['median_ms']:7.2f} ms (median) | p95 {b['p95_ms']:6.2f} ms | {b['fps']:6.1f} FPS")

SegFormer-B2 (teacher)       58.71 ms (median) | p95  60.41 ms |   17.0 FPS
SegFormer-B0 (direct)        40.77 ms (median) | p95  42.66 ms |   24.3 FPS
SegFormer-B0 (distilled)     40.84 ms (median) | p95  42.33 ms |   24.4 FPS
YOLO26n-sem (direct)         25.69 ms (median) | p95  26.40 ms |   38.8 FPS
YOLO26n-sem (distilled)      26.33 ms (median) | p95  27.02 ms |   37.9 FPS


## Where does the time actually go?

Same model, three stopwatches. The gaps between the columns tell you what each stage
costs — and how much a benchmark can flatter a model by starting the clock late.

In [ ]:
rows = []
for run, m in MODELS.items():
    b1, b2, b3 = bench1[run], bench2[run], bench3[run]
    rows.append({
        'model': m['display'],
        '1_raw_fwd_ms':    round(b1['median_ms'], 2),
        '2_full_640_ms':   round(b2['median_ms'], 2),
        '3_deployment_ms': round(b3['median_ms'], 2),
        'preprocess_cost_ms': round(b2['median_ms'] - b1['median_ms'], 2),
        'upsample_cost_ms':   round(b3['median_ms'] - b2['median_ms'], 2),
        '1_FPS': round(b1['fps'], 1), '2_FPS': round(b2['fps'], 1), '3_FPS': round(b3['fps'], 1),
        '3_p95_ms': round(b3['p95_ms'], 2),
    })
timing = pd.DataFrame(rows)
print('Benchmark 1 -> 2 gap = cost of preprocessing.  2 -> 3 gap = cost of upsampling to 24.2M px.')
print('Notice how much smaller the honest FPS (col 3) is than the architecture FPS (col 1).\n')
timing

Benchmark 1 -> 2 gap = cost of preprocessing.  2 -> 3 gap = cost of upsampling to 24.2M px.
Notice how much smaller the honest FPS (col 3) is than the architecture FPS (col 1).



,model,1_raw_fwd_ms,2_full_640_ms,3_deployment_ms,preprocess_cost_ms,upsample_cost_ms,1_FPS,2_FPS,3_FPS,3_p95_ms
0,SegFormer-B2 (teacher),33.67,44.96,58.71,11.29,13.75,29.7,22.1,17.0,60.41
1,SegFormer-B0 (direct),16.93,26.96,40.77,10.03,13.81,59.0,36.6,24.3,42.66
2,SegFormer-B0 (distilled),16.80,27.12,40.84,10.33,13.71,59.3,36.7,24.4,42.33
3,YOLO26n-sem (direct),8.48,12.47,25.69,3.99,13.22,117.2,80.2,38.8,26.40
4,YOLO26n-sem (distilled),8.51,12.23,26.33,3.73,14.09,117.0,80.6,37.9,27.02


---
# Accuracy evaluation — all 5 models on the 185-image test set at 640

Now we stop timing and measure **quality**. Every model is scored on the same 640×640
grid against the same ground truth, so the numbers are directly comparable.

**One important asymmetry.** SegFormer is fed the dataloader's tensor directly. YOLO is
**not** — we call Ultralytics' own `.predict()` on the original image file and then bring
its output down to 640. Why: `.predict()` applies YOLO's own letterbox + /255 internally,
which is how it was trained. Feeding it the dataloader's ImageNet-normalised tensor
instead scores **0.506 instead of 0.735 mean Dice** — the model is fine, the input recipe
was wrong. Ground truth comes from the dataloader in both cases, so the comparison stays fair.

**Metrics.** Dice/IoU measure overlap. **Complete-miss rate** is the fraction of images
with a bruise where the model predicts *nothing* — reported separately because a blank
mask is a missed injury, which is far worse than a slightly loose outline.

In [ ]:
def to_640_nearest(mask):
    """Any binary mask -> 640x640, nearest-neighbour. Squeeze guards against (H,W,1)."""
    m = np.asarray(mask)
    if m.ndim == 3:
        m = m.squeeze(-1)
    return (cv2.resize(m.astype('uint8'), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST) > 0).astype('uint8')

per_image = {}
for run, m in MODELS.items():
    rows = []
    for i, (x, y, stems, img_paths, _) in enumerate(test_loader, start=1):
        gt = (y[0, 0].numpy() > 0.5).astype('uint8')          # GT at 640, from the dataloader

        if m['family'] == 'segformer':
            with torch.no_grad():
                logits = m['model'](x.to(DEVICE))
            pred = postprocess_segformer(logits, m['threshold'])
        else:
            # Ultralytics native inference on the ORIGINAL file (its own letterbox + /255).
            res = m['wrapper'].predict(source=str(img_paths[0]), imgsz=IMG_H, device=0, verbose=False)[0]
            cm = res.semantic_mask.data                        # NOT res.masks (None for semantic models)
            if hasattr(cm, 'cpu'):
                cm = cm.cpu().numpy()
            pred = to_640_nearest((np.asarray(cm) == 1).astype('uint8'))   # native res -> 640

        rows.append(compute_image_row(pred, gt, str(stems[0])))
        if i % 50 == 0:
            print(f"  [{m['display']}] {i}/{len(test_df)}")

    df = pd.DataFrame(rows)
    df['complete_miss'] = (df.pred_positive_pixels == 0) & (df.gt_positive_pixels > 0)
    per_image[run] = df
    print(f"{m['display']:26s} median Dice={df.dice.median():.3f}  mean={df.dice.mean():.3f}  "
          f"miss={df.complete_miss.mean()*100:.2f}%")

  [SegFormer-B2 (teacher)] 50/185
  [SegFormer-B2 (teacher)] 100/185
  [SegFormer-B2 (teacher)] 150/185
SegFormer-B2 (teacher)     median Dice=0.827  mean=0.767  miss=0.00%
  [SegFormer-B0 (direct)] 50/185
  [SegFormer-B0 (direct)] 100/185
  [SegFormer-B0 (direct)] 150/185
SegFormer-B0 (direct)      median Dice=0.810  mean=0.775  miss=0.54%
  [SegFormer-B0 (distilled)] 50/185
  [SegFormer-B0 (distilled)] 100/185
  [SegFormer-B0 (distilled)] 150/185
SegFormer-B0 (distilled)   median Dice=0.828  mean=0.787  miss=0.54%
  [YOLO26n-sem (direct)] 50/185
  [YOLO26n-sem (direct)] 100/185
  [YOLO26n-sem (direct)] 150/185
YOLO26n-sem (direct)       median Dice=0.848  mean=0.735  miss=9.73%
  [YOLO26n-sem (distilled)] 50/185
  [YOLO26n-sem (distilled)] 100/185
  [YOLO26n-sem (distilled)] 150/185
YOLO26n-sem (distilled)    median Dice=0.843  mean=0.741  miss=7.03%


## Final results — accuracy and speed together

`FPS_deployment` is benchmark 3, the honest one.

Read the miss-rate column next to the Dice column: the model with the best Dice is not
necessarily the one you want in the field.

In [ ]:
rows = []
for run, m in MODELS.items():
    df = per_image[run]
    rows.append({
        'model': m['display'],
        'params_M': round(m['params_m'], 2),
        'threshold': 'argmax' if m['threshold'] is None else f"{m['threshold']:.2f}",
        'median_Dice': round(df.dice.median(), 4),
        'mean_Dice': round(df.dice.mean(), 4),
        'median_IoU': round(df.iou.median(), 4),
        'complete_miss_%': round(df.complete_miss.mean()*100, 2),
        'FPS_raw_fwd': round(bench1[run]['fps'], 1),
        'FPS_deployment': round(bench3[run]['fps'], 1),
    })
results = pd.DataFrame(rows)
results

,model,params_M,threshold,median_Dice,mean_Dice,median_IoU,complete_miss_%,FPS_raw_fwd,FPS_deployment
0,SegFormer-B2 (teacher),27.35,0.75,0.8271,0.7667,0.7052,0.00,29.7,17.0
1,SegFormer-B0 (direct),3.71,0.60,0.8100,0.7751,0.6807,0.54,59.0,24.3
2,SegFormer-B0 (distilled),3.71,0.55,0.8284,0.7872,0.7071,0.54,59.3,24.4
3,YOLO26n-sem (direct),1.63,argmax,0.8482,0.7353,0.7364,9.73,117.2,38.8
4,YOLO26n-sem (distilled),1.63,argmax,0.8425,0.7413,0.7279,7.03,117.0,37.9


## Save everything to Drive

Per-image CSVs (one row per test image per model), the results table, the 3-benchmark
timing table, and a metadata file recording exactly what each stopwatch included.

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = f'{LOCAL_DIR}/inference_demo'
os.makedirs(OUT, exist_ok=True)

for run, df in per_image.items():
    df.to_csv(f'{OUT}/per_image_{run}.csv', index=False)
results.to_csv(f'{OUT}/results_accuracy_and_speed.csv', index=False)
timing.to_csv(f'{OUT}/timing_three_benchmarks.csv', index=False)

json.dump({
  'gpu': torch.cuda.get_device_name(0),
  'n_test_images': len(test_df), 'scoring_grid': '640x640',
  'native_resolution': f'{NATIVE_W}x{NATIVE_H}',
  'timing_protocol': f'{N_WARMUP} warmup + {N_REPEATS} reps, cuda.synchronize() around each call',
  'benchmark_1_raw_forward': {
      'included': ['forward pass', 'sigmoid+threshold / argmax', 'mask to CPU'],
      'excluded': ['disk read', 'resize to 640', 'normalisation', 'host->GPU transfer',
                   'upsample to native'],
      'use_for': 'architecture comparison only'},
  'benchmark_2_full_640': {
      'included': ['resize to 640', 'normalisation/scaling', 'host->GPU transfer',
                   'forward pass', 'sigmoid+threshold / argmax', 'mask to CPU'],
      'excluded': ['disk read', 'upsample to native'],
      'use_for': 'inference cost at 640'},
  'benchmark_3_deployment': {
      'included': ['resize to 640', 'normalisation/scaling', 'host->GPU transfer',
                   'forward pass', 'sigmoid+threshold / argmax', 'mask to CPU',
                   'upsample mask 640 -> native 4022x6024'],
      'excluded': ['disk read/decode (camera frames arrive in RAM)', 'model loading',
                   'drawing the colour overlay'],
      'use_for': 'RECOMMENDED - the only number valid for deployment claims'},
  'segformer_preprocessing': 'stretch resize to 640 + ImageNet mean/std',
  'yolo_preprocessing': 'letterbox to 640 + /255 (no ImageNet norm)',
  'note_yolo': ('YOLO evaluated via Ultralytics native .predict(); feeding it the '
                'ImageNet-normalised dataloader tensor scores 0.506 vs 0.735 mean Dice.'),
}, open(f'{OUT}/metadata.json', 'w'), indent=2)

dest = f'{DRIVE_DIR}/inference_demo_{stamp}'
shutil.copytree(OUT, dest)
print('Saved to:', dest)
!ls -la {dest}

Saved to: /content/drive/MyDrive/bruise_segmentation_gpu/inference_demo_20260715_145229
total 128
-rw------- 1 root root  1644 Jul 15 14:52 metadata.json
-rw------- 1 root root 25963 Jul 15 14:52 per_image_segformer_b0_direct.csv
-rw------- 1 root root 25998 Jul 15 14:52 per_image_segformer_b0_distilled.csv
-rw------- 1 root root 25877 Jul 15 14:52 per_image_segformer_b2_teacher.csv
-rw------- 1 root root 24550 Jul 15 14:52 per_image_yolo_sem_direct.csv
-rw------- 1 root root 24839 Jul 15 14:52 per_image_yolo_sem_distilled.csv
-rw------- 1 root root   450 Jul 15 14:52 results_accuracy_and_speed.csv
-rw------- 1 root root   478 Jul 15 14:52 timing_three_benchmarks.csv
